In [ ]:
#this is a failed experment with a very basic sliding window which just aggregate the predictions with a basic function, a more premissing LSTM based aggregation is avilable
import pandas as pd
import numpy as np
from tqdm import tqdm_pandas
from tqdm.notebook import tqdm
from transformers import BertModel, BertTokenizerFast, Trainer, TrainingArguments, AutoModelForSequenceClassification, AutoTokenizer, BertForSequenceClassification, BertTokenizer
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support,  roc_auc_score, fbeta_score

import warnings
warnings.filterwarnings("ignore")


# hyperparameters
batch_size = 16
epochs = 3
learning_rate = 2e-5


model_name = 'onlplab/alephbert-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(model_name , num_labels=2)

checkpoint_path = "models/model_weights_progressive_curiculum_learning_conv_messages.pth"

# Load state dict
state_dict = torch.load(checkpoint_path, map_location="cpu")  # or "cuda" if on GPU

# Load weights into model
bert_model.load_state_dict(state_dict)

# Move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model.to(device)


/home/ronfr/.conda/envs/alephbert_eval/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at onlplab/alephbert-base and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(52000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [2]:
conv_info_path = 'trainDatasets/conv_info.csv'
conv_info_df = pd.read_csv(conv_info_path)
conv_info_df['engagement_id'] = conv_info_df['engagement_id'].astype(str)
conv_info_df = conv_info_df.rename(columns={'gsr': 'label'})
train_conv_df, test_conv_df = train_test_split(conv_info_df, test_size=0.2, stratify=conv_info_df['label'],random_state=42)

In [3]:
messages_with_lables_path = 'trainDatasets/combined_riskfree_riskfull_messages_syntatic_fixed.csv'

messages_with_lables_df = pd.read_csv(messages_with_lables_path)

messages_with_lables_df['engagement_id'] = messages_with_lables_df['engagement_id'].astype(str)
messages_with_lables_df = messages_with_lables_df[messages_with_lables_df['anonymized'].notna()]
messages_with_lables_df['name'] = messages_with_lables_df['name'].fillna('-')

#split to train and test base on conversation split
train_labled_messages_df = messages_with_lables_df[messages_with_lables_df["engagement_id"].isin(train_conv_df["engagement_id"])]
test_labled_messages_df = messages_with_lables_df[messages_with_lables_df["engagement_id"].isin(test_conv_df["engagement_id"])]

In [4]:
all_messages_path = 'trainDatasets/messages_anonymized.csv'

all_messages_df = pd.read_csv(all_messages_path)

all_messages_df['engagement_id'] = all_messages_df['engagement_id'].astype(str)
all_messages_df = all_messages_df[all_messages_df['anonymized'].notna()]
all_messages_df['name'] = all_messages_df['name'].fillna('-')
all_messages_df = all_messages_df[all_messages_df["seeker"]]

#split to train and test base on conversation split and concat messages
train_all_messages_df = all_messages_df[all_messages_df["engagement_id"].isin(train_conv_df["engagement_id"])]
train_all_messages_df = train_all_messages_df.merge(train_conv_df , on="engagement_id")
train_all_messages_df = train_all_messages_df.groupby('engagement_id').agg({'anonymized': '[SEP]'.join, 'label': 'first'}).reset_index()

test_all_messages_df = all_messages_df[all_messages_df["engagement_id"].isin(test_conv_df["engagement_id"])]
test_all_messages_df = test_all_messages_df.merge(test_conv_df , on="engagement_id")
test_all_messages_df = test_all_messages_df.groupby('engagement_id').agg({'anonymized': '[SEP]'.join, 'label': 'first'}).reset_index()


In [5]:
# mapping the text into inputs that fits the model
def tokenize(batch):
    return tokenizer(batch['anonymized'], padding='max_length', truncation=True, max_length=512)

def make_weighted_train_loaders(dfs, weights, tokenize):
    #make sampled df
    acc_train = pd.DataFrame(data= {})
    for df, weight in zip(dfs,weights):
        sample = df.sample(frac = weight)
        acc_train = pd.concat([acc_train,sample[["anonymized" , "label"]]])
    
    #create datasets
    train_dataset = Dataset.from_pandas(acc_train)
    train_dataset = train_dataset.map(tokenize, batched=True, batch_size=16)
    train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    return train_loader

def make_test_loader(df, tokenize):
    dataset = Dataset.from_pandas(df)
    dataset = dataset.map(tokenize, batched=True, batch_size=16)
    dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    test_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    return test_loader

In [6]:
#labled_messages_train_loader = make_weighted_train_loaders([train_labled_messages_df, train_all_messages_df] , [1,0], tokenize)
#all_messages_train_loader = make_weighted_train_loaders([train_labled_messages_df, train_all_messages_df] , [0,1], tokenize)
labled_messages_test_loader = make_test_loader(test_labled_messages_df, tokenize)
all_messages_test_loader = make_test_loader(test_all_messages_df, tokenize)

Map: 100%|██████████| 6047/6047 [00:04<00:00, 1214.54 examples/s]


In [7]:
def calc_matrics(labels, preds, pred_probs):
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    roc_auc = roc_auc_score(labels, pred_probs)
    f2 = fbeta_score(labels, preds, beta=2)
    
    print(f'Accuracy: {accuracy}')
    print(f'Precision: {precision}')
    print(f'Recall: {recall}')
    print(f'F1: {f1}')
    print(f'ROC-AUC: {roc_auc}')
    print(f'F2: {f2}')

In [ ]:
bert_model.eval()
labels = []
preds = []
pred_probs = []

for batch in labled_messages_test_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    label = batch['label'].to(device)

    with torch.no_grad():
        outputs = bert_model(input_ids, attention_mask=attention_mask)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)
    predictions = torch.argmax(logits, dim=-1)

    labels.extend(label.cpu().numpy())
    preds.extend(predictions.cpu().numpy())
    pred_probs.extend(probabilities[:, 1].cpu().numpy())

calc_matrics(labels,preds,pred_probs)

In [ ]:
bert_model.eval()
labels = []
preds = []
pred_probs = []

for batch in all_messages_test_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    label = batch['label'].to(device)

    with torch.no_grad():
        outputs = bert_model(input_ids, attention_mask=attention_mask)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)
    predictions = torch.argmax(logits, dim=-1)

    labels.extend(label.cpu().numpy())
    preds.extend(predictions.cpu().numpy())
    pred_probs.extend(probabilities[:, 1].cpu().numpy())

calc_matrics(labels,preds,pred_probs)

In [ ]:
class SlidingWindowDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, window_size=512, stride=256, label_column="label"):
        self.tokenizer = tokenizer
        self.samples = []
        self.labels = {}

        for idx, row in df.iterrows():
            text = row['anonymized']
            label = row[label_column]
            sample_id = row['engagement_id']  # this will serve as grouping key

            # Tokenize without truncation
            enc = tokenizer(text, truncation=False, return_tensors="pt")
            input_ids = enc["input_ids"][0]

            for start in range(0, len(input_ids), stride):
                end = min(start + window_size, len(input_ids))
                if end - start < 10:
                    continue

                chunk = input_ids[start:end]
                self.samples.append({
                    "input_ids": chunk,
                    "sample_id": sample_id
                })
                self.labels[sample_id] = label

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        padded = self.tokenizer.pad(
            {"input_ids": [sample["input_ids"]]},
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )

        return {
            "input_ids": padded["input_ids"].squeeze(0),
            "attention_mask": padded["attention_mask"].squeeze(0),
            "sample_id": sample["sample_id"]
        }


In [ ]:
test_dataset = SlidingWindowDataset(test_all_messages_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [ ]:
from collections import defaultdict
import torch.nn.functional as F

bert_model.eval()

logits_per_sample = defaultdict(list)
labels_per_sample = {}

for batch in test_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    sample_ids = batch['sample_id']

    with torch.no_grad():
        outputs = bert_model(input_ids, attention_mask=attention_mask)

    logits = outputs.logits

    for i in range(len(sample_ids)):
        sample_id = sample_ids[i]
        logits_per_sample[sample_id].append(logits[i].cpu())

        # Save true label once (assuming all chunks have the same label)
        if sample_id not in labels_per_sample:
            labels_per_sample[sample_id] = test_all_messages_df[
                test_all_messages_df["engagement_id"] == sample_id
            ]["label"].iloc[0]



In [25]:
# ---------- Aggregation ----------
labels = []
preds = []
pred_probs = []

for sample_id, logits_list in logits_per_sample.items():
    k = min(3, len(probs_list))
    topk_probs = torch.topk(torch.stack(probs_list)[:, 1], k).values
    avg_topk = topk_probs.mean()
    pred = int(avg_topk > 0.5)

    preds.append(pred)
    pred_probs.append(prob[1].item())  # probability of class 1
    labels.append(labels_per_sample[sample_id])

# ---------- Metrics ----------
calc_matrics(labels, preds, pred_probs)

Accuracy: 0.8188441794999172
Precision: 0.0
Recall: 0.0
F1: 0.0
ROC-AUC: 0.5
F2: 0.0
